# FASTA -> ProtT5 Embeddings -> BioData Best Neighbors

Generate ProtT5 embeddings for all proteins in a FASTA file and report the best BioData neighbor for each query protein.

## Notes

- This notebook uses `CBBIO.embeddings.Generator` (factory) with `model_class="protT5"` for generation.
- The generator returns **per-residue** embeddings. Pooling is intentionally done here (user-side).
- The notebook excludes the query protein ID from neighbor search, so the reported hit is the best non-self match.
- You need a running BioData PostgreSQL instance configured via `config.yaml` or `BIODATA_*` env vars.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import numpy as np
import pandas as pd
from time import perf_counter

from CBBIO.BioData import BioDataClient, NotFoundError
from CBBIO.embeddings import load_fasta_inputs
from CBBIO import (
    EmbeddingWriter,
    Generator,
    IterableBatcher,
    pooler_factory,
    run_embedding_generation,
)
from tqdm.auto import tqdm


import torch

device='cpu'
if(torch.cuda.is_available()):
    print("cuda avaible:", torch.cuda.get_device_name(0))
    device="cuda"
if torch.backends.mps.is_available():
    device = "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

In [ ]:
# User inputs
FASTA_CANDIDATES = [
    PROJECT_ROOT / "random_protein_ids.fasta",
    Path.cwd() / "random_protein_ids.fasta",
]
FASTA_PATH = next((path for path in FASTA_CANDIDATES if path.exists()), FASTA_CANDIDATES[0])
MAX_RECORDS = None  # set an int to limit records during experimentation
MAX_PROTEIN_LENGTH = 1000  # skip proteins longer than this many amino acids
BATCH_SIZE = 1  # set None for token batching only
MAX_BATCH_TOKENS = None  # set an int for token-budget batching
REQUESTED_LAYERS = [0]  # Native hidden-state indexing: 0 is the earliest hidden state
POOLING = "mean"

EMBEDDING_TYPE_NAME = "Prot-T5"  # must exist in sequence_embedding_type.name


if not FASTA_PATH.exists():
    raise FileNotFoundError(f"FASTA file not found: {FASTA_PATH}")

fasta_inputs = load_fasta_inputs(FASTA_PATH)
if MAX_RECORDS is not None:
    fasta_inputs = fasta_inputs[:MAX_RECORDS]
original_record_count = len(fasta_inputs)
fasta_inputs = [record for record in fasta_inputs if len(record.sequence) <= MAX_PROTEIN_LENGTH]
filtered_out_count = original_record_count - len(fasta_inputs)

if not fasta_inputs:
    raise ValueError(
        f"No FASTA records <= {MAX_PROTEIN_LENGTH} aa found in {FASTA_PATH}"
    )

input_ids = [record.id for record in fasta_inputs]
sequence_lengths = {record.id: len(record.sequence) for record in fasta_inputs}

print({
    "fasta_path": str(FASTA_PATH),
    "record_count": len(fasta_inputs),
    "filtered_out_too_long": filtered_out_count,
    "max_protein_length": MAX_PROTEIN_LENGTH,
    "first_query_id": input_ids[0],
    "first_sequence_len": sequence_lengths[input_ids[0]],
})


## Generate ProtT5 Embeddings

In [ ]:
batch_count = None
sequence_aa_total = sum(sequence_lengths.values())

generator = Generator(model_class=EMBEDDING_TYPE_NAME, device=device)
batcher = IterableBatcher(
    fasta_inputs,
    batch_size=BATCH_SIZE,
    max_batch_tokens=MAX_BATCH_TOKENS,
)
writer = EmbeddingWriter(format="memory")
progress_events = []

generation_started_at = perf_counter()
job_result = run_embedding_generation(
    generator,
    batcher,
    writer,
    layer_index=REQUESTED_LAYERS,
    pooler=pooler_factory("mean"),
    fail_fast=False,
    progress_callback=progress_events.append,
)
generation_elapsed_seconds = perf_counter() - generation_started_at

generated_records = writer.records
generation_errors = job_result.errors
model_metadata = getattr(generator, "model_metadata", None)
batch_elapsed_seconds = []

if generation_errors:
    raise RuntimeError(
        f"Embedding generation failed for {job_result.error_count} records. "
        f"First errors: {generation_errors[:3]}"
    )

benchmark_summary = {
    "device": device,
    "sequence_count": len(fasta_inputs),
    "total_amino_acids": sequence_aa_total,
    "generation_elapsed_seconds": generation_elapsed_seconds,
    "proteins_per_second": len(fasta_inputs) / generation_elapsed_seconds,
    "amino_acids_per_second": sequence_aa_total / generation_elapsed_seconds,
    "benchmark_excludes_model_load": True,
}

print(model_metadata)
print({
    "sequence_count": len(fasta_inputs),
    "batch_size": BATCH_SIZE,
    "max_batch_tokens": MAX_BATCH_TOKENS,
    "generated_records": len(generated_records),
    "requested_layers": REQUESTED_LAYERS,
    "resolved_layers": sorted({int(record.layer_index) for record in generated_records}),
})
print(benchmark_summary)

# Old generation loop kept for transition reference:
#
# batch_size = int(globals().get("BATCH_SIZE", 1))
# if batch_size != 1:
#     raise ValueError("BATCH_SIZE must be 1 for this notebook.")
#
# batch_count = (len(fasta_inputs) + batch_size - 1) // batch_size
# generated_records = []
# generation_errors = []
# model_metadata = None
# batch_elapsed_seconds = []
# generation_started_at = perf_counter()
#
# for start in tqdm(range(0, len(fasta_inputs), batch_size), desc="Generating embeddings", unit="protein"):
#     batch_inputs = fasta_inputs[start:start + batch_size]
#     batch_started_at = perf_counter()
#     batch_result = generator.generate(
#         batch_inputs,
#         layer_index=REQUESTED_LAYERS,
#     )
#     batch_elapsed_seconds.append(perf_counter() - batch_started_at)
#     if model_metadata is None:
#         model_metadata = batch_result.model_metadata
#     generated_records.extend(batch_result.records)
#     generation_errors.extend(batch_result.errors)


## Protein level pooling

In [ ]:
# Pick one generated layer for DB lookup. Records are already mean-pooled by the job pooler.
target_layer = REQUESTED_LAYERS[0] if REQUESTED_LAYERS else generated_records[0].layer_index
layer_records = [record for record in generated_records if record.layer_index == target_layer]
if not layer_records:
    raise RuntimeError(f"Layer {target_layer} was not generated.")

pooled_embeddings = {}
pooling_rows = []

for record in layer_records:
    pooled = np.asarray(record.embedding, dtype=np.float32)
    if pooled.ndim != 1:
        raise RuntimeError(f"Expected 1D pooled embedding for {record.id}, got shape={pooled.shape}")

    pooled_embeddings[record.id] = pooled
    pooling_rows.append({
        "query_id": record.id,
        "layer": int(record.layer_index),
        "sequence_len": int(sequence_lengths[record.id]),
        "pooled_dim": int(pooled.shape[0]),
        "embedding_fingerprint": float(pooled[:10].sum()),
        "embedding_l2": float(np.linalg.norm(pooled)),
    })

missing_ids = [query_id for query_id in input_ids if query_id not in pooled_embeddings]
if missing_ids:
    raise RuntimeError(f"Missing pooled embeddings for: {missing_ids[:10]}")

pooling_df = pd.DataFrame(pooling_rows).sort_values("query_id").reset_index(drop=True)

print({
    "layer": int(target_layer),
    "pooled_records": len(pooling_df),
    "pooling": POOLING,
    "pooled_dim": int(pooling_df.iloc[0]["pooled_dim"]),
})

pooling_df.head()


## Find Best Neighbors In The DB

In [ ]:
# BioData lookup configuration

TOP_K = 1
METRIC = "cosine"  # l2, cosine, inner_product

# Adapter convention: the protT5 generator uses native hidden-state layer numbering.
layer_index_in_db = int(target_layer)

# create DB client and connect
client = BioDataClient()
client.connect()

emb_type = client.get_embedding_type_by_name(EMBEDDING_TYPE_NAME)
if emb_type is None:
    raise NotFoundError(
        f"Embedding type '{EMBEDDING_TYPE_NAME}' not found in BioData. "
        "Check sequence_embedding_type table or change EMBEDDING_TYPE_NAME."
    )

print({
    "embedding_type_id": emb_type.id,
    "embedding_type_name": emb_type.name,
    "generated_layer": int(target_layer),
    "db_layer_index": int(layer_index_in_db),
    "query_count": len(input_ids),
})

best_neighbor_rows = []
best_neighbor_ids = []

for query_id in input_ids:
    query_embedding = pooled_embeddings[query_id]
    neighbors = client.find_nearest_neighbors(
        query_embedding.tolist(),
        embedding_type_id=emb_type.id,
        layer_index=layer_index_in_db,
        k=TOP_K,
        metric=METRIC,
        exclude_protein_ids=[query_id],
    )

    if not neighbors:
        best_neighbor_rows.append({
            "query_id": query_id,
            "query_sequence_len": sequence_lengths[query_id],
            "best_neighbor_id": None,
            "distance": None,
            "neighbor_layer_index": None,
            "neighbor_go_terms": 0,
        })
        continue

    best_neighbor = neighbors[0]
    best_neighbor_ids.append(best_neighbor.protein_id)
    best_neighbor_rows.append({
        "query_id": query_id,
        "query_sequence_len": sequence_lengths[query_id],
        "best_neighbor_id": best_neighbor.protein_id,
        "distance": best_neighbor.distance,
        "neighbor_layer_index": best_neighbor.layer_index,
    })

annotations = client.fetch_go_annotations(best_neighbor_ids)

for row in best_neighbor_rows:
    neighbor_id = row["best_neighbor_id"]
    row["neighbor_go_terms"] = len(annotations.get(neighbor_id, [])) if neighbor_id else 0

best_neighbors_df = pd.DataFrame(best_neighbor_rows).sort_values(
    by=["distance", "query_id"],
    na_position="last",
).reset_index(drop=True)

best_neighbors_df


In [ ]:
print({
    "query_count": len(input_ids),
    "neighbors_found": int(best_neighbors_df["best_neighbor_id"].notna().sum()),
    "mean_distance": float(best_neighbors_df["distance"].dropna().mean()),
})

best_neighbors_df.head(10)


In [ ]:
client.close()